In [ ]:
#Agent avec réponse structurée en utilisant BaseModel
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_ollama import ChatOllama

In [ ]:
# Initialiser le modèle Ollama
model = ChatOllama(
model="llama3.2:3b", # ou mistral, gemma, etc.
temperature=0
)

In [ ]:
#Création du Tool
class CapitalInfo(BaseModel):
  nom: str
  Localisation: str
  Ambiance: str
  Économie: str

In [4]:
system_prompt = """

Vous êtes un auteur de science-fiction et vous devez créer une capitale spa-
tiale à la demande d'un utilisateur.

Veuillez respecter la structure ci-dessous.
Nom : Nom de la capitale
Localisation : Lieu où elle est située
Ambiance : Description en 2 ou 3 mots
Économie : Principaux secteurs d'activité
"""

In [8]:
agent = create_agent(
model=model,
system_prompt=system_prompt,
response_format=CapitalInfo
)
question = HumanMessage(content="Quelle est la capitale de la lune ?")
response = agent.invoke(
{"messages": [question]}
)
response["structured_response"]

CapitalInfo(nom='Capitole Lunaire', Localisation='Lune', Ambiance='Astronomique', Économie='Tourisme Spatiale')

In [11]:
#Création du Tool
from langchain.tools import tool
@tool("meteo_capitale")
def meteo_capitale(ville: str) -> str:
    """
    Donne la météo d'une capitale (valeurs fixes pour test).
    Args:
        ville: nom de la capitale
    """
    print("tool meteo_capitale utilisé")
    temperature = 25
    humidite = 60
    pression = 1013

    return (
        f"Météo à {ville} : "
        f"Température = {temperature}°C, "
        f"Humidité = {humidite}%, "
        f"Pression = {pression} hPa")

In [12]:
#Ajout du Tool à l’agent
system_prompt = "Utilises les tools pour répondre aux questions"
agent = create_agent(
model=model,
tools=[meteo_capitale],
system_prompt=system_prompt,
)
question = HumanMessage(content="Quelle est la météo à Capitole lunaire ?")
response = agent.invoke(
{"messages": [question]}
)
print(response['messages'][-1].content)

tool meteo_capitale utilisé
Voici la réponse formatée :

La météo à Capitole lunaire est actuellement :

* Température : 25°C
* Humidité : 60%
* Pression : 1013 hPa

J'espère que cela répond à votre question ! Si vous avez besoin de plus d'informations, n'hésitez pas à me demander.
